# ADME public data preparation

---

We used a collection of 3,521 diverse compounds selected from commercially available compound libraries (i.e. Enamine, eMolecules, WuXi LabNetwork, Mcule) and tested against six ADME in vitro assays: HLM, RLM, Solubility, MDR1-MDCK ER, hPPB, and rPPB.

Some basic preprocessing steps, such as data transformations, were applied, data distributions were examined, and a scaffold split is performed.

Source: https://github.com/molecularinformatics/Computational-ADME.git - Dataset to download: 'ADME_public_set_3521.csv'

**Note:** This notebook must be run using the `unique-env` conda environment. The scaffold-based split of the preprocessed data is performed separately in `Split_data.ipynb`, which must be run using the `chemprop-env` conda environment.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
# from rdkit import Chem
# from chemprop.data.splitting import make_split_indices
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

In [2]:
# Load data
inputFile = "data/ADME_public_set_3521.csv"
df = pd.read_csv(inputFile)
df.head()

,Internal ID,Vendor ID,SMILES,CollectionName,LOG HLM_CLint (mL/min/kg),LOG MDR1-MDCK ER (B-A/A-B),LOG SOLUBILITY PH 6.8 (ug/mL),LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),LOG PLASMA PROTEIN BINDING (RAT) (% unbound),LOG RLM_CLint (mL/min/kg)
0,Mol1,317714313,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,emolecules,0.675687,1.493167,0.089905,0.991226,0.518514,1.392169
1,Mol2,324056965,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,emolecules,0.675687,1.040780,0.550228,0.099681,0.268344,1.027920
2,Mol3,304005766,CN(c1ncc(F)cn1)[C@H]1CCCNC1,emolecules,0.675687,-0.358806,NaN,2.000000,2.000000,1.027920
3,Mol4,194963090,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,emolecules,0.675687,1.026662,1.657056,-1.158015,-1.403403,1.027920
4,Mol5,324059015,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,emolecules,0.996380,1.010597,NaN,1.015611,1.092264,1.629093


In [3]:
df.shape

(3521, 10)

In [4]:
# Remove disconnected SMILES
ixx_keep = ['.' not in x for x in df['SMILES']]
df_filtered = df.iloc[ixx_keep]
df_filtered.shape

(3517, 10)

In [5]:
# Convert logPPB to logfu
df_filtered.loc[:,'LOG Fu (HUMAN)'] = [np.nan if np.isnan(x) else x - 2 for x in df_filtered['LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)']]
df_filtered.loc[:,'LOG Fu (RAT)'] = [np.nan if np.isnan(x) else x - 2 for x in df_filtered['LOG PLASMA PROTEIN BINDING (RAT) (% unbound)']]
df_filtered.head()

/tmp/ipykernel_406880/2720915221.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.loc[:,'LOG Fu (HUMAN)'] = [np.nan if np.isnan(x) else x - 2 for x in df_filtered['LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)']]
/tmp/ipykernel_406880/2720915221.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.loc[:,'LOG Fu (RAT)'] = [np.nan if np.isnan(x) else x - 2 for x in df_filtered['LOG PLASMA PROTEIN BINDING (RAT) (% unbound)']]


,Internal ID,Vendor ID,SMILES,CollectionName,LOG HLM_CLint (mL/min/kg),LOG MDR1-MDCK ER (B-A/A-B),LOG SOLUBILITY PH 6.8 (ug/mL),LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),LOG PLASMA PROTEIN BINDING (RAT) (% unbound),LOG RLM_CLint (mL/min/kg),LOG Fu (HUMAN),LOG Fu (RAT)
0,Mol1,317714313,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,emolecules,0.675687,1.493167,0.089905,0.991226,0.518514,1.392169,-1.008774,-1.481486
1,Mol2,324056965,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,emolecules,0.675687,1.040780,0.550228,0.099681,0.268344,1.027920,-1.900319,-1.731656
2,Mol3,304005766,CN(c1ncc(F)cn1)[C@H]1CCCNC1,emolecules,0.675687,-0.358806,NaN,2.000000,2.000000,1.027920,0.000000,0.000000
3,Mol4,194963090,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,emolecules,0.675687,1.026662,1.657056,-1.158015,-1.403403,1.027920,-3.158015,-3.403403
4,Mol5,324059015,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,emolecules,0.996380,1.010597,NaN,1.015611,1.092264,1.629093,-0.984389,-0.907736


## Data distributions

In [ ]:
# assays_summary = pd.DataFrame()
# for c in df_filtered.columns[-8:]:
#     assays_summary = assays_summary.append(df_filtered[c].describe())
# assays_summary = assays_summary.sort_values('count', ascending=False)
# assays_summary

In [ ]:
# for c in df_filtered.columns[-8:]:
#     plt.figure(figsize=(4, 3))
#     sns.histplot(data=df_filtered[df_filtered[c].notna()], x=c)

In [7]:
df_filtered.shape

(3517, 12)

## Final dataset

In [8]:
# Rename endpoints and select columns for final dataset
df_filtered = df_filtered.rename(columns={'Internal ID': 'Id',
                                           'SMILES': 'Structure',
                                           'LOG RLM_CLint (mL/min/kg)': 'rLM LogCLint', 'LOG HLM_CLint (mL/min/kg)': 'hLM LogCLint',
                                           'LOG MDR1-MDCK ER (B-A/A-B)':'MDCK-MDR1_LogER',
                                           'LOG Fu (RAT)': 'LogFu-Rat', 'LOG Fu (HUMAN)': 'LogFu-Human'})

In [9]:
keep_cols = ['Id', 'Structure', 'rLM LogCLint', 'hLM LogCLint', 'MDCK-MDR1_LogER', 'LogFu-Rat','LogFu-Human']
df_filtered = df_filtered.loc[:, keep_cols]
df_filtered = df_filtered.reset_index(drop=True)
df_filtered.head()

,Id,Structure,rLM LogCLint,hLM LogCLint,MDCK-MDR1_LogER,LogFu-Rat,LogFu-Human
0,Mol1,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,1.392169,0.675687,1.493167,-1.481486,-1.008774
1,Mol2,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,1.027920,0.675687,1.040780,-1.731656,-1.900319
2,Mol3,CN(c1ncc(F)cn1)[C@H]1CCCNC1,1.027920,0.675687,-0.358806,0.000000,0.000000
3,Mol4,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,1.027920,0.675687,1.026662,-3.403403,-3.158015
4,Mol5,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,1.629093,0.996380,1.010597,-0.907736,-0.984389


## Save file

Timestamp for saved files

In [10]:
from datetime import datetime
timestamp_file = datetime.now().strftime("%Y%m%d")
timestamp_file

'20260908'

In [11]:
outputFileName = f'{inputFile.split(".csv")[0]}_preprocessed{timestamp_file}.csv'
outputFileName

'data/ADME_public_set_3521_preprocessed20260908.csv'

In [12]:
df_filtered.to_csv(outputFileName, index=False)